In [2]:
import os
import numpy as np
import scipy.sparse as spp
import dgl
import torch
import h5py

In [3]:
spe = "yeast"

action_file = os.path.join("data", spe, "action/protein.action.tsv")
embedding_h5 = os.path.join("data", spe, "seq/protein.embedding.h5")
cmap_dir = os.path.join("data", spe, "cmap")

In [4]:
ids = set()
with open(action_file, "r") as fin:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        ss = line.split("\t")
        ids.add(ss[0])
        ids.add(ss[1])

print(f"Number of protein IDs: {len(ids)}")

Number of protein IDs: 2497


In [5]:
embed_data = {}
with h5py.File(embedding_h5, "r") as h5fin:
    for id in h5fin.keys():
        embed_data[id] = h5fin[id][:, :]

In [6]:
for id in ids:
    embedding = embed_data[id]
    cmap_file = os.path.join(cmap_dir, id + ".npz")
    cmap_data = np.load(cmap_file)
    node_num = len(str(cmap_data["seq"]))
    cmap = cmap_data["contact"]
    g_embed = torch.tensor(embedding[:node_num]).float()
    adj = spp.coo_matrix(cmap)
    G = dgl.from_scipy(adj)
    
    if G.num_nodes() != g_embed.shape[0]:
        print(f"Protein {id} - {G.num_nodes()} : {g_embed.shape[0]}")